# 9.13 Reward modeling

Reward modeling converts human preferences into a scalar training signal.

This notebook implements the Bradley-Terry pairwise loss with small linear rewards. D5 includes reward gaming, so the metric is pairwise preference accuracy checked against held-out human outcomes.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 913
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)


def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))


def build_reward_ladder():
    rng = np.random.default_rng(SEED)
    ladders = []
    ladders.append({"name": "D1 one preference", "chosen": np.array([1.5]), "rejected": np.array([0.0]), "human": np.array([1])})
    ladders.append({"name": "D2 few-shot pairs", "chosen": np.array([1.2, 0.7, 1.6, 0.2, 0.9]), "rejected": np.array([0.1, 0.2, 0.5, 0.4, 0.0]), "human": np.ones(5)})
    ladders.append({"name": "D3 annotator artifacts", "chosen": np.array([1.0, 0.4, 1.3, 0.2, 1.5, 0.6]), "rejected": np.array([0.2, 0.8, 0.3, 0.5, 0.7, 0.1]), "human": np.ones(6)})
    ladders.append({"name": "D4 helpfulness pairs", "chosen": np.array([1.4, 1.1, 0.8, 1.5, 0.9, 1.2, 0.6, 1.0]), "rejected": np.array([0.5, 0.4, 0.7, 0.2, 0.1, 1.0, 0.3, 0.4]), "human": np.ones(8)})
    chosen_d5 = rng.normal(loc=1.0, scale=0.6, size=50)
    rejected_d5 = rng.normal(loc=0.4, scale=0.6, size=50)
    chosen_d5[::8] = -0.2
    rejected_d5[::8] = 2.2
    human_d5 = np.ones(50)
    ladders.append({"name": "D5 reward gaming", "chosen": chosen_d5, "rejected": rejected_d5, "human": human_d5, "hacks": np.arange(0, 50, 8)})
    return ladders


## The concept, built once (D1)

The reward-model loss is $$\mathcal{L}_{\mathrm{RM}}=-\log \sigma(r_\theta(x,y^+)-r_\theta(x,y^-)).$$
The exact lesson checks are: gap $1.5$ gives probability $0.818$ and loss $0.201$; reversed gap $-1.0$ gives probability $0.269$ and loss $1.313$; adding $+5$ to both rewards leaves probability unchanged; and losses $0.201,0.693,1.313$ average to $0.736$.

In [ ]:

def reward_model_loss(chosen_rewards, rejected_rewards):
    chosen_rewards = np.asarray(chosen_rewards, dtype=float)
    rejected_rewards = np.asarray(rejected_rewards, dtype=float)
    gaps = chosen_rewards - rejected_rewards
    probabilities = sigmoid(gaps)
    losses = -np.log(probabilities + 1e-12)
    return probabilities, losses, float(np.mean(losses))

prob_demo, loss_demo, mean_demo = reward_model_loss(np.array([1.5]), np.array([0.0]))
prob_bad, loss_bad, mean_bad = reward_model_loss(np.array([0.0]), np.array([1.0]))
prob_shifted, loss_shifted, mean_shifted = reward_model_loss(np.array([6.5]), np.array([5.0]))
average_demo = np.mean([0.201, 0.693, 1.313])
assert round(float(prob_demo[0]), 3) == 0.818
assert round(float(loss_demo[0]), 3) == 0.201
assert round(float(prob_bad[0]), 3) == 0.269
assert round(float(loss_bad[0]), 3) == 1.313
assert round(float(prob_shifted[0]), 3) == 0.818
assert round(float(average_demo), 3) == 0.736
print(prob_demo[0], loss_demo[0], prob_bad[0], loss_bad[0], average_demo)


Only reward differences matter. We compute pairwise preference accuracy from whether chosen reward exceeds rejected reward.

In [ ]:

chosen = np.array([1.5, 0.0, 0.3])
rejected = np.array([0.0, 0.0, 1.3])
probabilities, losses, mean_loss = reward_model_loss(chosen, rejected)
preference_accuracy = float(np.mean(chosen > rejected))
print("probabilities", probabilities)
print("losses", losses)
print("pairwise accuracy", preference_accuracy)


## The dataset ladder

The ladder grows from one preference pair to D5 reward gaming examples, where model scores disagree with held-out human outcomes.

In [ ]:

ladder = build_reward_ladder()
for rung in ladder:
    print(rung["name"], "pairs", len(rung["chosen"]), "chosen mean", np.mean(rung["chosen"]), "rejected mean", np.mean(rung["rejected"]))
    print("sample gaps", (rung["chosen"] - rung["rejected"])[:5])


## Run the same reward loss across D1-D5

In [ ]:

results = []
for rung in ladder:
    probabilities, losses, mean_loss = reward_model_loss(rung["chosen"], rung["rejected"])
    accuracy = float(np.mean(rung["chosen"] > rung["rejected"]))
    results.append({"name": rung["name"], "probabilities": probabilities, "losses": losses, "metric": accuracy, "mean_loss": mean_loss, "chosen": rung["chosen"], "rejected": rung["rejected"]})

print("rung | pairwise_accuracy | mean_loss")
for result in results:
    print(f"{result['name']} | {result['metric']:.3f} | {result['mean_loss']:.3f}")


## Results visualization

Small multiples show chosen and rejected rewards. The summary curve tracks pairwise preference accuracy.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for ax, result in zip(axes, results):
    count = min(10, len(result["chosen"]))
    positions = np.arange(count)
    ax.plot(positions, result["chosen"][:count], marker="o", label="chosen")
    ax.plot(positions, result["rejected"][:count], marker="x", label="rejected")
    ax.set_title(result["name"].split()[0])
    ax.set_xlabel("pair")
    ax.set_ylabel("reward")
axes[0].legend()
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(results) + 1), [item["metric"] for item in results], marker="o")
ax.set_xticks(range(1, len(results) + 1))
ax.set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
ax.set_ylabel("pairwise preference accuracy")
ax.set_title("Reward model quality")
plt.show()


## Pitfall on the hardest rung

Pitfall: high reward can be gamed. On D5, the rejected answer sometimes receives higher model reward; a held-out human outcome check catches it.

In [ ]:

d5 = ladder[-1]
probabilities, losses, mean_loss = reward_model_loss(d5["chosen"], d5["rejected"])
model_prefers_chosen = d5["chosen"] > d5["rejected"]
raw_accuracy = float(np.mean(model_prefers_chosen == d5["human"].astype(bool)))
hack_rows = d5["hacks"]
hack_accuracy = float(np.mean(model_prefers_chosen[hack_rows] == d5["human"][hack_rows].astype(bool)))
nonhack = np.ones(len(model_prefers_chosen), dtype=bool)
nonhack[hack_rows] = False
fixed_accuracy = float(np.mean(model_prefers_chosen[nonhack] == d5["human"][nonhack].astype(bool)))
print("raw D5 accuracy", raw_accuracy)
print("reward-hack subset accuracy", hack_accuracy)
print("held-out nonhack accuracy", fixed_accuracy)


## Evaluate it + Practice

- Metric: pairwise preference accuracy, with mean Bradley-Terry loss as calibration.
- No-skill baseline: random pair ordering gives about 50% accuracy and loss near $0.693$ for zero gaps.
- Cheap sanity check: gap $1.5$ must give probability $0.818$ and loss $0.201$.
- Ablation: add a constant to all rewards and confirm probabilities do not change.
- Failure signals: high model reward on hacked answers, annotator artifacts, or good loss with poor human outcomes.

Practice prompts:


**Practice.** Add +5 to both D5 reward arrays and verify pairwise probabilities stay fixed.

**Practice.** Increase reward-hack rows and observe raw D5 accuracy.

**Practice.** Calibrate a threshold for uncertain pairs with probabilities near 0.5.